# Agentic Default-Payments Pipeline — Demo

End-to-end walkthrough of the CrewAI pipeline:

1. **DataAgent** loads the Taiwan default-of-credit-card-clients CSV and emits a JSON summary.
2. **TrainerAgent** trains Random Forest, XGBoost, and an MLP and emits a JSON metric report.
3. **ExplainerAgent** turns that report into a plain-English interpretability brief.

Set `GEMINI_API_KEY` in your environment (or in `../../../.env`) before running the agent cells.

In [ ]:
import os, sys, pathlib, json

PROJECT_ROOT = pathlib.Path().resolve().parents[0]
SRC = PROJECT_ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

try:
    from dotenv import load_dotenv
    load_dotenv(PROJECT_ROOT.parents[1] / '.env')
except Exception:
    pass

print('GEMINI_API_KEY set:', bool(os.environ.get('GEMINI_API_KEY')))

## 1. Sanity-check the ML layer (no LLM calls)

Confirm the data loader and trainers work before involving the agents.

In [ ]:
from agentic_default.data_loader import load_dataset, csv_to_json_records
from agentic_default.ml_trainer import train_and_evaluate

ds = load_dataset()
print('rows:', ds.metadata.n_rows, '|', 'features:', ds.metadata.n_features)
print('class balance:', ds.metadata.class_balance)
print('feature names:', ds.feature_names[:6], '...')

preview = csv_to_json_records(sample=2)
print('first JSON record:', json.dumps(preview[0], indent=2))

In [ ]:
report = train_and_evaluate(
    ds.x_train, ds.y_train, ds.x_test, ds.y_test,
    feature_names=ds.feature_names,
    models=['random_forest', 'xgboost', 'neural_network'],
)
for row in report['leaderboard']:
    print(row)
print('best model:', report['best_model'])

## 2. Run the full agentic pipeline

Requires `GEMINI_API_KEY`.

In [ ]:
from agentic_default.pipeline import run_pipeline

run = run_pipeline(
    models=['random_forest', 'xgboost', 'neural_network'],
    output_dir=str(PROJECT_ROOT / 'outputs' / 'demo_run'),
)
print('--- Dataset summary ---')
print(json.dumps(run.dataset_summary, indent=2)[:600])
print('\n--- Leaderboard ---')
print(json.dumps(run.metrics_report.get('leaderboard', []), indent=2))

In [ ]:
from IPython.display import Markdown
Markdown(run.explanation_markdown)

## 3. Inspect the persisted artifacts

Everything the agents produced is also written to disk for reproducibility.

In [ ]:
out_dir = PROJECT_ROOT / 'outputs' / 'demo_run'
for f in sorted(out_dir.iterdir()):
    print(f.name, '-', f.stat().st_size, 'bytes')